In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Installation


In [ ]:
!pip install rouge-score pycocoevalcap

# Lib


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import re
import ast
import math
import glob
import copy
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import kagglehub
from PIL import Image
from tqdm import tqdm
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import CosineAnnealingLR

from transformers import AutoTokenizer, AutoModel

import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from pycocoevalcap.cider.cider import Cider
from rouge_score import rouge_scorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

# Data Preparation


## Utils

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def print_directory_structure(startpath, max_level=3):
    start_level = startpath.count(os.sep)

    for root, dirs, files in os.walk(startpath):
        level = root.count(os.sep) - start_level
        if level > max_level:
            del dirs[:]
            continue

        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')

        if level < max_level:
            subindent = ' ' * 4 * (level + 1)
            for f in files:
                print(f'{subindent}{f}')

def parse_list_string(val):
    if pd.isna(val):
        return []
    try:
        parsed = ast.literal_eval(val)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

def select_best_single_image(row):
    pa_list = parse_list_string(row["PA"])
    if pa_list:
        return pa_list[0]

    ap_list = parse_list_string(row["AP"])
    if ap_list:
        return ap_list[0]

    img_list = parse_list_string(row["image"])
    if img_list:
        return img_list[0]

    return None

def extract_all_images_from_row(row):
    all_imgs = []

    pa_list = parse_list_string(row["PA"])
    if pa_list:
        all_imgs.extend(pa_list)

    ap_list = parse_list_string(row["AP"])
    if ap_list:
        all_imgs.extend(ap_list)

    img_list = parse_list_string(row["image"])
    if img_list:
        all_imgs.extend(img_list)

    return list(set(all_imgs))

def filter_existing_images(df, root_dir, image_col='best_image'):
    exists_mask = []
    for img_path in tqdm(df[image_col]):
        if pd.isna(img_path):
            exists_mask.append(False)
            continue
        full_path = os.path.join(root_dir, img_path)
        exists_mask.append(os.path.exists(full_path))
    filtered_df = df[exists_mask].reset_index(drop=True)
    return filtered_df

def clean_stringified_list(val):
    if pd.isna(val):
        return val
    try:
        parsed_list = ast.literal_eval(val)
        if isinstance(parsed_list, list) and len(parsed_list) > 0:
            return parsed_list[0]
        return val
    except (ValueError, SyntaxError):
        return val

## Dowload

In [ ]:
path = kagglehub.dataset_download("simhadrisadaram/mimic-cxr-dataset")
DATA_DIR_ROOT = path
print("Path to dataset files:", path)

In [ ]:
#print_directory_structure(path)

In [ ]:
DATA_DIR = path
ROOT_DIR = os.path.join(DATA_DIR_ROOT, "official_data_iccv_final")
TRAIN_CSV = os.path.join(DATA_DIR, "mimic_cxr_aug_train.csv")
VAL_CSV = os.path.join(DATA_DIR, "mimic_cxr_aug_validate.csv")

print(f"DATA_DIR: {DATA_DIR}")
print(f"TRAIN CSV: {TRAIN_CSV}")
print(f"VAL CSV: {VAL_CSV}")

## Splitting

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
df_raw = pd.concat([train_df, val_df], ignore_index=True)

In [ ]:
df_raw = df_raw.sample(frac=1, random_state=42).reset_index(drop=True)

final_records = []
seen_images = set()
target_size = 72000

for row in df_raw.to_dict("records"):
    img_list = parse_list_string(row.get("image", ""))
    raw_view = row.get("view", "")
    raw_text = row.get("text", "")
    parsed_text = parse_list_string(raw_text)

    findings_text = ""
    for section in parsed_text:
        if "findings" in section.lower():
            findings_text = section
            break

    if not findings_text:
        continue

    for img in img_list:
        if img in seen_images:
            continue

        img_path = os.path.join(ROOT_DIR, img)
        if os.path.exists(img_path):
            seen_images.add(img)

            sub_id = row.get(
                "subject_id",
                row.get("Subject_id", row.get("SUBJECT_ID", None)),
            )

            final_records.append(
                {
                    "subject_id": sub_id,
                    "view": raw_view,
                    "best_image": img,
                    "path": img,
                    "text": findings_text,
                }
            )

        if len(seen_images) >= target_size:
            break

    if len(seen_images) >= target_size:
        break

df_all = pd.DataFrame(final_records)
print(f"Total images: {len(df_all)}")

Total images: 72000


In [ ]:
grouped_patients = [
    patient_df
    for _, patient_df in df_all.groupby("subject_id")
]

rng = np.random.default_rng(42)
rng.shuffle(grouped_patients)


TRAIN_TARGET = 50000
VAL_TARGET = 10000
TEST_TARGET = 10000

train_chunks = []
val_chunks = []
test_chunks = []

train_count = 0
val_count = 0
test_count = 0


for patient_df in grouped_patients:

    n_imgs = len(patient_df)

    if train_count < TRAIN_TARGET:
        train_chunks.append(patient_df)
        train_count += n_imgs

    elif val_count < VAL_TARGET:
        val_chunks.append(patient_df)
        val_count += n_imgs

    elif test_count < TEST_TARGET:
        test_chunks.append(patient_df)
        test_count += n_imgs

    else:
        break


train_df_final = pd.concat(train_chunks, ignore_index=True)
val_df_final = pd.concat(val_chunks, ignore_index=True)
test_df_final = pd.concat(test_chunks, ignore_index=True)

train_subjects = set(train_df_final["subject_id"])
val_subjects = set(val_df_final["subject_id"])
test_subjects = set(test_df_final["subject_id"])

print("Train ∩ Val :", len(train_subjects & val_subjects))
print("Train ∩ Test:", len(train_subjects & test_subjects))
print("Val ∩ Test  :", len(val_subjects & test_subjects))
print()
print(
    f"Train : {len(train_df_final):,} images | "
    f"{len(train_subjects):,} patients"
)

print(
    f"Val   : {len(val_df_final):,} images | "
    f"{len(val_subjects):,} patients"
)

print(
    f"Test  : {len(test_df_final):,} images | "
    f"{len(test_subjects):,} patients"
)

Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0
Train : 50,001 images | 8,714 patients
Val   : 10,003 images | 1,690 patients
Test  : 10,006 images | 1,716 patients


In [ ]:
train_ids = set(train_df_final["subject_id"])
val_ids = set(val_df_final["subject_id"])
test_ids = set(test_df_final["subject_id"])

print(len(train_ids & val_ids))
print(len(train_ids & test_ids))
print(len(val_ids & test_ids))

0
0
0


In [ ]:
train_df_final.to_csv("/content/drive/MyDrive/UEH TERMS/Kì 6/Tutorial Deep Learning/colab/data/mimic_train.csv", index=False)
val_df_final.to_csv("/content/drive/MyDrive/UEH TERMS/Kì 6/Tutorial Deep Learning/colab/data/mimic_val.csv", index=False)
test_df_final.to_csv("/content/drive/MyDrive/UEH TERMS/Kì 6/Tutorial Deep Learning/colab/data/mimic_test.csv", index=False)

In [ ]:
train_df_final.head()

,subject_id,view,best_image,path,text
0,14895473,"['PA', 'LL']",files/p14/p14895473/s51189039/0dd3b044-f3e826c...,files/p14/p14895473/s51189039/0dd3b044-f3e826c...,Findings: PA and lateral chest views were obta...
1,14895473,"['PA', 'LL']",files/p14/p14895473/s51189039/9a85c547-c9a0696...,files/p14/p14895473/s51189039/9a85c547-c9a0696...,Findings: PA and lateral chest views were obta...
2,19996786,"['LL', 'PA', 'LATERAL']",files/p19/p19996786/s52281280/543da826-2aa513c...,files/p19/p19996786/s52281280/543da826-2aa513c...,"Findings: In comparison with the study of ___,..."
3,19996786,"['LL', 'PA', 'LATERAL']",files/p19/p19996786/s52281280/5df00fef-29f8142...,files/p19/p19996786/s52281280/5df00fef-29f8142...,"Findings: In comparison with the study of ___,..."
4,19996786,"['LL', 'PA', 'LATERAL']",files/p19/p19996786/s52281280/ff766df0-156bec0...,files/p19/p19996786/s52281280/ff766df0-156bec0...,"Findings: In comparison with the study of ___,..."


## Calculation

In [ ]:
batch_size = 64
num_workers = 4

dummy_vocab = Vocabulary(freq_threshold=1)
dummy_vocab.build_vocabulary(['dummy caption'])

train_dataset_for_stats = MIMICCXRDataset(df=train_df_final, root_dir=ROOT_DIR, transform=val_transform, vocab=dummy_vocab)
train_loader_for_stats = DataLoader(
    dataset=train_dataset_for_stats,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    collate_fn=MyCollate(pad_idx=dummy_vocab.stoi["<PAD>"])
)

mean_sum = torch.zeros(3)
std_sum = torch.zeros(3)
num_pixels = 0

print("Calculating mean and std of the dataset...")
for images, _, _ in tqdm(train_loader_for_stats):
    images = images.cpu()
    images = images.view(images.size(0), images.size(1), -1)
    mean_sum += images.sum(2).sum(0)
    std_sum += (images**2).sum(2).sum(0)
    num_pixels += images.size(0) * images.size(2)

mean = mean_sum / num_pixels
var = (std_sum / num_pixels) - (mean ** 2)
std = torch.sqrt(var)

print(f"Calculated Mean: {mean.tolist()}")
print(f"Calculated Std: {std.tolist()}")

Calculating mean and std of the dataset...


  0%|          | 0/782 [00:00<?, ?it/s]

Calculated Mean: [-0.056120723485946655, 0.07209093868732452, 0.293992817401886]
Calculated Std: [1.3214750289916992, 1.3509714603424072, 1.3449667692184448]
